# Week 23, Day 5: Strange attractors, Lyapunov exponents and chaotic networks

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w023/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The largest Lyapunov exponent

After 10 time units of settling (RK4, dt = 0.01), follow a second trajectory starting 1e-8 away in x. At every step add log(d / d0) to a running sum and pull the second trajectory back to distance d0 along the separation. Over 10,000 steps, store the estimate in `lyap_lorenz` and the main trajectory in `traj`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
sig, rho, bet = 10.0, 28.0, 8 / 3
lor = lambda p: np.array([sig * (p[1] - p[0]), p[0] * (rho - p[2]) - p[1], p[0] * p[1] - bet * p[2]])
def rk4(f, x, dt):
    k1 = f(x); k2 = f(x + dt / 2 * k1); k3 = f(x + dt / 2 * k2); k4 = f(x + dt * k3)
    return x + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)

x, dt = np.array([1.0, 1.0, 20.0]), 0.01
for _ in range(1000): x = rk4(lor, x, dt)
d0 = 1e-8; y = x + np.array([d0, 0, 0]); S = 0.0; n = 10000; traj = np.empty((n, 3))
for k in range(n):
    x, y = rk4(lor, x, dt), rk4(lor, y, dt)
    d = np.linalg.norm(y - x)
    S += ...
    y = ...                                      # back to distance d0, same direction
    traj[k] = x
lyap_lorenz = S / (n * dt)
print(f"largest Lyapunov exponent about {lyap_lorenz:.3f} per time unit")
plt.plot(traj[:, 0], traj[:, 2], lw=0.4); plt.xlabel("x"); plt.ylabel("z"); plt.show()

In [ ]:
import numpy as np
def test_lyap():
    "the estimate is within 0.08 of the accepted 0.906"
    assert abs(lyap_lorenz - 0.906) < 0.08

def test_bounded():
    "the trajectory stays bounded (|x| below 30 and z between 0 and 60)"
    assert np.abs(traj[:, 0]).max() < 30 and traj[:, 2].min() > 0 and traj[:, 2].max() < 60

test_lyap()
test_bounded()
print("All checks passed")

### Exercise: Lorenz's map of maxima

From `traj`, find the local maxima of z (points higher than both neighbours) and store their values in `zmax`. For each z_n (except the last), find the nearest other z_m and use z_{m+1} as a prediction of z_{n+1}; store the median absolute error in `pred_err`.

In [ ]:
z = traj[:, 2]
idx = np.where((z[1:-1] > z[:-2]) & (z[1:-1] > z[2:]))[0] + 1
zmax = z[idx]
zn, zn1 = zmax[:-1], zmax[1:]
errs = []
for k in range(len(zn)):
    dist = np.abs(zn - zn[k]); dist[k] = np.inf
    j = ...
    errs.append(abs(zn1[j] - zn1[k]))
pred_err = ...
print(f"{len(zmax)} maxima;  nearest-neighbour prediction error {pred_err:.3f};  spread of z_(n+1) {np.std(zn1):.2f}")
plt.plot(zn, zn1, ".", ms=3); plt.plot([30, 48], [30, 48], "k:"); plt.xlabel("z_n"); plt.ylabel("z_(n+1)"); plt.show()

In [ ]:
import numpy as np
def test_many():
    "there are more than 100 maxima"
    assert len(zmax) > 100

def test_map():
    "the next maximum is predicted from its nearest neighbour's successor with median error below 0.3, while z_{n+1} itself varies with sd above 2"
    assert pred_err < 0.3 and np.std(zmax[1:]) > 2

test_many()
test_map()
print("All checks passed")

### Exercise: From silence to chaos

For g = 0.5 and 3, draw J (N = 200, entries of sd g/√N) and a random start, run 1,000 Euler steps of 0.1 to settle, then 2,000 more while tracking a second trajectory started 1e-8 away (renormalising as in Step 2). Store the final standard deviation of x across neurons in `act_sd[g]` and the exponent in `lyap_net[g]`.

In [ ]:
act_sd, lyap_net, traces = {}, {}, {}
for g in (0.5, 3.0):
    r_g = np.random.default_rng(1988); N = 200
    J = r_g.normal(0, g / np.sqrt(N), (N, N))
    Fn = lambda v: -v + J @ np.tanh(v)
    x = r_g.normal(size=N); dt = 0.1
    for _ in range(1000): x = x + dt * Fn(x)
    y = x + 1e-8 * r_g.normal(size=N) / np.sqrt(N); d0 = np.linalg.norm(y - x); S = 0.0; tr = []
    for _ in range(2000):
        x, y = x + dt * Fn(x), y + dt * Fn(y)
        d = np.linalg.norm(y - x)
        S += np.log(d / d0); y = x + (y - x) * d0 / d
        tr.append(x[:3].copy())
    act_sd[g] = ...
    lyap_net[g] = ...
    traces[g] = np.array(tr)
print({g: f"{v:.3g}" for g, v in act_sd.items()}, " activity sd;", {g: round(float(v), 3) for g, v in lyap_net.items()}, " Lyapunov exponent")
plt.plot(np.arange(2000) * 0.1, traces[3.0]); plt.xlabel("time (units of tau)"); plt.ylabel("x of three neurons, g = 3"); plt.show()

In [ ]:
import numpy as np
def test_quiet():
    "for g = 0.5 activity dies out (sd below 1e-6) and the Lyapunov exponent is negative"
    assert act_sd[0.5] < 1e-6 and lyap_net[0.5] < 0

def test_chaos():
    "for g = 3 activity is sustained (sd above 1) and the Lyapunov exponent is positive (above 0.05)"
    assert act_sd[3.0] > 1 and lyap_net[3.0] > 0.05

test_quiet()
test_chaos()
print("All checks passed")